In [1]:
import pymupdf


In [2]:
document_path = "../data/documents/document_textuel.pdf"

document = pymupdf.open(document_path)

print("Nombre de pages :", len(document))

Nombre de pages : 18


In [3]:
document_full_text = ""

for page in document:
    document_full_text += page.get_text()

In [4]:
page = document[0]

page_text = page.get_text()

print(page_text)

Résumé pour tous
Changement 
Climatique 2021:



In [5]:
print("Nombre de caractères :", len(document_full_text))

Nombre de caractères : 40598


In [6]:
print(document_full_text[:3000])

Résumé pour tous
Changement 
Climatique 2021:
WORKING GROUP I TECHNICAL SUPPORT UNIT
Ce document n’a pas fait l’objet de la procédure de relecture officielle du GIEC.
• Changement Climatique 2021 : Résumé pour tous •
3 /
Météo, climat et GIEC 
Quel que soit l’endroit où nous habitons, nous vivons tous avec la météo : comment les conditions 
de notre atmosphère évoluent au fil des minutes, des heures, des jours et des semaines. Nous 
vivons également tous avec le climat, c’est-à-dire, en un lieu donné, l’ensemble des caractéristiques 
météorologiques moyennes sur plusieurs décennies. On parle de changement climatique lorsque 
ces conditions moyennes commencent à se modifier, du fait de causes naturelles ou du fait des 
activités humaines. La hausse des températures, les variations des précipitations, l’intensification de 
phénomènes météorologiques extrêmes sont autant d’exemples de changements climatiques, parmi 
bien d’autres caractéristiques. 
En 1990, le premier rapport du Groupe d’

In [7]:
import re 

In [8]:
def clean_text(text):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    return text.strip()

In [9]:
cleaned_document_text = clean_text(document_full_text)

In [10]:
len(cleaned_document_text)

40579

In [11]:
print(cleaned_document_text[:3000])

Résumé pour tous
Changement 
Climatique 2021:
WORKING GROUP I TECHNICAL SUPPORT UNIT
Ce document n’a pas fait l’objet de la procédure de relecture officielle du GIEC.
• Changement Climatique 2021 : Résumé pour tous •
3 /
Météo, climat et GIEC 
Quel que soit l’endroit où nous habitons, nous vivons tous avec la météo : comment les conditions 
de notre atmosphère évoluent au fil des minutes, des heures, des jours et des semaines. Nous 
vivons également tous avec le climat, c’est-à-dire, en un lieu donné, l’ensemble des caractéristiques 
météorologiques moyennes sur plusieurs décennies. On parle de changement climatique lorsque 
ces conditions moyennes commencent à se modifier, du fait de causes naturelles ou du fait des 
activités humaines. La hausse des températures, les variations des précipitations, l’intensification de 
phénomènes météorologiques extrêmes sont autant d’exemples de changements climatiques, parmi 
bien d’autres caractéristiques. 
En 1990, le premier rapport du Groupe d’

CHUNKER

In [23]:
def create_chunks(text, chunk_size=2000, chunk_overlap=300):
    paragraphs = text.split("\n\n")

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        if len(paragraph) <= chunk_size:

            if len(current_chunk) + len(paragraph) + 2 <= chunk_size:
                current_chunk += paragraph + "\n\n"

            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())

                current_chunk = paragraph + "\n\n"

        else:

            if current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = ""

            start = 0

            while start < len(paragraph):

                end = start + chunk_size

                if end < len(paragraph):
                    split_position = paragraph.rfind(" ", start, end)

                    if split_position != -1:
                        end = split_position

                chunk = paragraph[start:end]

                chunks.append(chunk.strip())

                start = end - chunk_overlap

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [24]:
chunks = create_chunks(
    cleaned_document_text,
    chunk_size=2000,
    chunk_overlap=300
)

print("Nombre de chunks :", len(chunks))

Nombre de chunks : 29


In [25]:
print("Fin du Chunk 1 :")
print(chunks[0][-300:])

print("\nDébut du Chunk 2 :")
print(chunks[1][:300])

Fin du Chunk 1 :
lions d’années (par exemple : 
cernes d’arbres, carottages dans les roches ou les glaces) ;
•  Des modèles informatiques qui explorent les changements passés, actuels et futurs (voir encadré 
Que sont les modèles climatiques ? à la page 9) ;
•  La compréhension du fonctionnement du climat (processus

Début du Chunk 2 :
lions d’années (par exemple : 
cernes d’arbres, carottages dans les roches ou les glaces) ;
•  Des modèles informatiques qui explorent les changements passés, actuels et futurs (voir encadré 
Que sont les modèles climatiques ? à la page 9) ;
•  La compréhension du fonctionnement du climat (processus


In [27]:
from sentence_transformers import SentenceTransformer

In [28]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [29]:
chunk_embeddings = model.encode(chunks)

print("Nombre de chunks :", len(chunk_embeddings))
print("Dimension d'un embedding :", len(chunk_embeddings[0]))

Nombre de chunks : 29
Dimension d'un embedding : 384


In [30]:
print(chunk_embeddings[0][:10])  # Affiche les 10 premières dimensions du premier embedding

[-0.05406011 -0.02523498  0.14687514 -0.00654908  0.06077958 -0.04411289
 -0.01380601  0.05798112  0.00732099  0.08101668]


In [31]:
query = "Pourquoi les glaciers diminuent-ils ?"

In [32]:
query_embedding = model.encode([query])

print(query_embedding.shape)

(1, 384)


In [33]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    query_embedding,
    chunk_embeddings
)

print(similarities.shape)

(1, 29)


In [34]:
print(similarities[0])

[0.34473506 0.35081786 0.43541163 0.6040548  0.5526297  0.4138895
 0.31384963 0.27506775 0.3475835  0.3649308  0.31961688 0.34856316
 0.3502643  0.36629146 0.25082743 0.30185843 0.32350248 0.34096536
 0.38278612 0.43551937 0.31901875 0.4279738  0.4299638  0.28082296
 0.30615374 0.3526137  0.3762191  0.33061558 0.30513543]


In [35]:
top_k = 3

top_indices = similarities[0].argsort()[-top_k:][::-1]

print(top_indices)

[ 3  4 19]


In [36]:
for index in top_indices:
    print(f"\n--- CHUNK {index + 1} ---")
    print("Score :", similarities[0][index])
    print(chunks[index])


--- CHUNK 4 ---
Score : 0.6040548
us élevées 
afin de suivre les déplacements des zones climatiques.
•  La saison de croissance de certaines espèces végétales de l’hémisphère nord s’est allongée (jusqu’à 
quatorze jours de plus depuis les années 1950) et, dans l’ensemble, la surface des terres est devenue plus 
verte depuis le début des années 1980.
Les glaces
•  De nombreuses zones englacées de la Terre fondent ou dégèlent rapidement. Globalement, les chutes 
de neige diminuent. Le recul généralisé des glaciers constaté depuis 1950 est inédit sur plus de 2000 ans.
•  La superficie de l’océan Arctique couverte de banquise en été est aujourd’hui 40 % plus réduite que 
dans les années 1980. Elle est la plus réduite depuis au moins mille ans.
•  Le manteau neigeux de l’hémisphère nord a diminué depuis la fin des années 1970, et certains sols 
habituellement gelés toute l’année se sont réchauffés et ont dégelé.

--- CHUNK 5 ---
Score : 0.5526297
Pour en savoir 
plus sur le 
changement 
cl

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, chunks, chunk_embeddings, model, top_k=3):

    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []

    for index in top_indices:
        results.append({
            "chunk_id": index,
            "score": similarities[index],
            "text": chunks[index]
        })

    return results

In [38]:
query = "Pourquoi les glaciers diminuent-ils ?"

results = retrieve(
    query,
    chunks,
    chunk_embeddings,
    model,
    top_k=3
)

for result in results:
    print(f"\n--- CHUNK {result['chunk_id'] + 1} ---")
    print(f"Score : {result['score']:.4f}")
    print(result["text"])


--- CHUNK 4 ---
Score : 0.6041
us élevées 
afin de suivre les déplacements des zones climatiques.
•  La saison de croissance de certaines espèces végétales de l’hémisphère nord s’est allongée (jusqu’à 
quatorze jours de plus depuis les années 1950) et, dans l’ensemble, la surface des terres est devenue plus 
verte depuis le début des années 1980.
Les glaces
•  De nombreuses zones englacées de la Terre fondent ou dégèlent rapidement. Globalement, les chutes 
de neige diminuent. Le recul généralisé des glaciers constaté depuis 1950 est inédit sur plus de 2000 ans.
•  La superficie de l’océan Arctique couverte de banquise en été est aujourd’hui 40 % plus réduite que 
dans les années 1980. Elle est la plus réduite depuis au moins mille ans.
•  Le manteau neigeux de l’hémisphère nord a diminué depuis la fin des années 1970, et certains sols 
habituellement gelés toute l’année se sont réchauffés et ont dégelé.

--- CHUNK 5 ---
Score : 0.5526
Pour en savoir 
plus sur le 
changement 
climatiq

In [39]:
queries = [
    "Pourquoi les glaciers diminuent-ils ?",
    "Comment les activités humaines modifient-elles le climat ?",
    "Que sont les modèles climatiques ?",
    "Comment les océans sont-ils affectés par le changement climatique ?"
]

for query in queries:

    print("\n" + "=" * 80)
    print("QUESTION :", query)
    print("=" * 80)

    results = retrieve(query, chunks, chunk_embeddings, model, top_k=2)

    for result in results:
        print(
            f"\nChunk {result['chunk_id'] + 1} "
            f"| Score : {result['score']:.4f}"
        )
        print(result["text"][:500])


QUESTION : Pourquoi les glaciers diminuent-ils ?

Chunk 4 | Score : 0.6041
us élevées 
afin de suivre les déplacements des zones climatiques.
•  La saison de croissance de certaines espèces végétales de l’hémisphère nord s’est allongée (jusqu’à 
quatorze jours de plus depuis les années 1950) et, dans l’ensemble, la surface des terres est devenue plus 
verte depuis le début des années 1980.
Les glaces
•  De nombreuses zones englacées de la Terre fondent ou dégèlent rapidement. Globalement, les chutes 
de neige diminuent. Le recul généralisé des glaciers constaté depuis 

Chunk 5 | Score : 0.5526
Pour en savoir 
plus sur le 
changement 
climatique, vous 
pouvez lire 
la Foire aux 
questions (FAQs) 
du rapport ici : 
https://www.ipcc.ch/report/ar6/wg1/
resources/frequently-asked-questions/
• Changement Climatique 2021 : Résumé pour tous •
5 /
•  Les calottes glaciaires du Groenland et de l’Antarctique se réduisent, tout comme la grande majorité 
des glaciers à travers le monde, ce qui aj

In [40]:
results = retrieve(
    query,
    chunks,
    chunk_embeddings,
    model,
    top_k=3
)

In [41]:
context = "\n\n".join(
    result["text"]
    for result in results
)

print(context)

Pour en savoir 
plus sur le 
changement 
climatique, vous 
pouvez lire 
la Foire aux 
questions (FAQs) 
du rapport ici : 
https://www.ipcc.ch/report/ar6/wg1/
resources/frequently-asked-questions/
• Changement Climatique 2021 : Résumé pour tous •
5 /
•  Les calottes glaciaires du Groenland et de l’Antarctique se réduisent, tout comme la grande majorité 
des glaciers à travers le monde, ce qui ajoute d’énormes quantités d’eau à l’océan. 
L’océan
•  90 % de la chaleur supplémentaire associée au réchauffement planétaire se sont accumulés dans 
l’océan (voir encadré Que sont les gaz à effet de serre ? à la page 6). L’océan se réchauffe aujourd’hui 
plus rapidement qu’à n’importe quel autre moment des derniers 11 000 ans.
•  Le niveau global de la mer s’est élevé d’environ vingt centimètres (environ huit pouces) depuis 1900. 
Cette élévation est plus rapide que sur toute autre période des derniers 3 000 ans au moins, et son 
rythme s’accélère.
•  En absorbant le dioxyde de carbone contenu da

In [42]:
prompt = f"""
Tu es un assistant qui répond aux questions en utilisant uniquement
les informations fournies dans le contexte.

Si la réponse n'est pas présente dans le contexte,
dis que l'information n'est pas disponible.

Contexte :
{context}

Question :
{query}

Réponse :
"""

In [43]:
print(prompt)


Tu es un assistant qui répond aux questions en utilisant uniquement
les informations fournies dans le contexte.

Si la réponse n'est pas présente dans le contexte,
dis que l'information n'est pas disponible.

Contexte :
Pour en savoir 
plus sur le 
changement 
climatique, vous 
pouvez lire 
la Foire aux 
questions (FAQs) 
du rapport ici : 
https://www.ipcc.ch/report/ar6/wg1/
resources/frequently-asked-questions/
• Changement Climatique 2021 : Résumé pour tous •
5 /
•  Les calottes glaciaires du Groenland et de l’Antarctique se réduisent, tout comme la grande majorité 
des glaciers à travers le monde, ce qui ajoute d’énormes quantités d’eau à l’océan. 
L’océan
•  90 % de la chaleur supplémentaire associée au réchauffement planétaire se sont accumulés dans 
l’océan (voir encadré Que sont les gaz à effet de serre ? à la page 6). L’océan se réchauffe aujourd’hui 
plus rapidement qu’à n’importe quel autre moment des derniers 11 000 ans.
•  Le niveau global de la mer s’est élevé d’environ v

In [44]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [45]:
response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": "Explique simplement ce qu'est le changement climatique."
        }
    ]
)

print(response.choices[0].message.content)

Le changement climatique, c’est le réchauffement progressif de la Terre et les modifications du climat qui en résultent.

En simplifié :

- Le Soleil réchauffe la Terre.
- Une partie de cette chaleur repart vers l’espace.
- Certains gaz dans l’atmosphère, comme le CO₂, retiennent une partie de cette chaleur : c’est l’effet de serre.
- Depuis longtemps, cet effet de serre existe naturellement et permet à la vie d’exister.
- Mais depuis le XIXᵉ siècle, les humains en rejettent beaucoup trop, surtout en brûlant du charbon, du pétrole et du gaz.
- Résultat : la planète se réchauffe plus vite.

Cela provoque des conséquences comme :

- des vagues de chaleur plus fréquentes,
- la fonte des glaces,
- la montée du niveau de la mer,
- des sécheresses,
- des inondations,
- des tempêtes parfois plus fortes,
- des perturbations pour les animaux, les plantes et les humains.

En résumé : le changement climatique, c’est quand le climat de la Terre change à cause surtout des activités humaines, et cel

In [46]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

print("Clé chargée :", api_key is not None)

Clé chargée : True


In [47]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

In [48]:
response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": "Explique simplement ce qu'est le changement climatique."
        }
    ]
)

print(response.choices[0].message.content)

Le changement climatique désigne le réchauffement progressif de la Terre et les modifications des patterns climatiques qui en résultent, principalement causés par l’augmentation des gaz à effet de serre (comme le CO₂) dans l’atmosphère. Ces gaz retiennent la chaleur du soleil, ce qui élève la température moyenne de la planète, modifie les régimes de précipitations, influence les événements météorologiques extrêmes et affecte les écosystèmes et les sociétés humaines.


In [49]:
response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response.choices[0].message.content

print(answer)



D'après le contexte fourni, les océans sont affectés par le changement climatique de quatre manières principales :

1.  **Réduction des glaces et augmentation du volume d'eau** : Les calottes glaciaires du Groenland et de l'Antarctique se réduisent, tout comme la grande majorité des glaciers, ce qui ajoute d'énormes quantités d'eau à l'océan.
2.  **Réchauffement** : L'océan a accumulé 90 % de la chaleur supplémentaire associée au réchauffement planétaire. Il se réchauffe aujourd'hui plus rapidement qu'à n'importe quel autre moment des derniers 11 000 ans.
3.  **Élévation du niveau de la mer** : Le niveau global de la mer s'est élevé d'environ vingt centimètres depuis 1900, un rythme plus rapide que sur toute autre période des derniers 3 000 ans au moins, et qui s'accélère.
4.  **Acidification** : En absorbant le dioxyde de carbone de l'atmosphère, l'océan s'acidifie. L'acidification des eaux de surface est maintenant exceptionnelle par rapport aux deux derniers millions d'années.


In [50]:
query = "Que sont les modèles climatiques ?"

In [51]:
results = retrieve(
    query,
    chunks,
    chunk_embeddings,
    model,
    top_k=3
)

context = "\n\n".join(
    result["text"]
    for result in results
)

prompt = f"""
Tu es un assistant qui répond aux questions en utilisant uniquement
les informations fournies dans le contexte.

Si la réponse n'est pas présente dans le contexte,
dis que l'information n'est pas disponible.

Contexte :
{context}

Question :
{query}

Réponse :
"""

In [52]:
response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

Les modèles climatiques sont des outils informatiques que les scientifiques utilisent pour comprendre les changements climatiques passés, présents et futurs. Ce sont des programmes informatiques qui simulent le climat de la Terre, sur la base des lois fondamentales de la physique, de la chimie et de la biologie de l'atmosphère, de l'océan, de la glace et des terres. Certains modèles incluent plus de processus, de complexité et de détails que d'autres. Ainsi, les simulations climatiques qui en résultent peuvent varier d'un modèle à l'autre ; c'est pourquoi le GIEC examine toujours les résultats de nombreux modèles climatiques afin de comprendre quels sont les résultats associés à notre plus grand degré de certitude.


In [53]:
def rag_answer(query, top_k=3):

    # 1. Retrieval
    results = retrieve(
        query,
        chunks,
        chunk_embeddings,
        model,
        top_k=top_k
    )

    # 2. Construction du contexte
    context = "\n\n".join(
        result["text"]
        for result in results
    )

    # 3. Construction du prompt
    prompt = f"""
Tu es un assistant qui répond aux questions en utilisant uniquement
les informations fournies dans le contexte.

Si la réponse n'est pas présente dans le contexte,
dis que l'information n'est pas disponible.

Contexte :
{context}

Question :
{query}

Réponse :
"""

    # 4. Génération avec le LLM
    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    # 5. Retourner la réponse
    return response.choices[0].message.content

In [54]:
answer = rag_answer(
    "Que sont les modèles climatiques ?"
)

print(answer)

Les modèles climatiques sont des outils informatiques utilisés par les scientifiques pour comprendre les changements climatiques passés, présents et futurs. Ils simulent le climat de la Terre en s'appuyant sur les lois fondamentales de la physique, de la chimie et de la biologie des systèmes atmosphériques, océaniques, glaciaires et terrestres. Ces modèles varient en complexité et en détails, ce qui entraîne des simulations différentes selon les outils employés. Le GIEC analyse les résultats de plusieurs modèles pour identifier les conclusions les plus fiables. Les modèles sont validés en comparant leurs prédictions aux observations historiques et aux données paléoclimatiques, ce qui renforce leur utilisation pour identifier les causes des changements climatiques et étudier leurs évolutions futures, notamment en fonction des émissions de gaz à effet de serre et des scénarios humains.


In [55]:
pages = []

for i, page in enumerate(textual_doc):

    text = page.get_text()

    pages.append({
        "page": i + 1,
        "text": text
    })

print("Nombre de pages :", len(pages))

NameError: name 'textual_doc' is not defined

In [ ]:
print(pages[0]["page"])
print(pages[0]["text"][:500])

1
Résumé pour tous
Changement 
Climatique 2021:



In [ ]:
def create_chunks_with_pages(pages, chunk_size=2000, chunk_overlap=300):

    chunks = []

    for page in pages:

        page_number = page["page"]
        text = clean_text(page["text"])

        paragraphs = text.split("\n\n")

        current_chunk = ""

        for paragraph in paragraphs:

            if len(paragraph) <= chunk_size:

                if len(current_chunk) + len(paragraph) + 2 <= chunk_size:
                    current_chunk += paragraph + "\n\n"

                else:
                    if current_chunk:
                        chunks.append({
                            "text": current_chunk.strip(),
                            "page": page_number
                        })

                    current_chunk = paragraph + "\n\n"

            else:

                if current_chunk:
                    chunks.append({
                        "text": current_chunk.strip(),
                        "page": page_number
                    })
                    current_chunk = ""

                start = 0

                while start < len(paragraph):

                    end = start + chunk_size

                    if end < len(paragraph):

                        split_position = paragraph.rfind(
                            " ",
                            start,
                            end
                        )

                        if split_position != -1:
                            end = split_position

                    chunk = paragraph[start:end]

                    chunks.append({
                        "text": chunk.strip(),
                        "page": page_number
                    })

                    start = end - chunk_overlap

        if current_chunk:
            chunks.append({
                "text": current_chunk.strip(),
                "page": page_number
            })

    return chunks

In [ ]:
chunks_with_pages = create_chunks_with_pages(
    pages,
    chunk_size=2000,
    chunk_overlap=300
)

print("Nombre de chunks :", len(chunks_with_pages))

Nombre de chunks : 40


In [ ]:
print(chunks_with_pages[0])

{'text': 'Résumé pour tous\nChangement \nClimatique 2021:', 'page': 1}


In [ ]:
for i in range(5):

    print(f"\n--- CHUNK {i + 1} ---")
    print("Page :", chunks_with_pages[i]["page"])
    print(chunks_with_pages[i]["text"][:300])


--- CHUNK 1 ---
Page : 1
Résumé pour tous
Changement 
Climatique 2021:

--- CHUNK 2 ---
Page : 2
WORKING GROUP I TECHNICAL SUPPORT UNIT
Ce document n’a pas fait l’objet de la procédure de relecture officielle du GIEC.

--- CHUNK 3 ---
Page : 3
• Changement Climatique 2021 : Résumé pour tous •
3 /
Météo, climat et GIEC 
Quel que soit l’endroit où nous habitons, nous vivons tous avec la météo : comment les conditions 
de notre atmosphère évoluent au fil des minutes, des heures, des jours et des semaines. Nous 
vivons également tous avec le 

--- CHUNK 4 ---
Page : 3
t futurs (voir encadré 
Que sont les modèles climatiques ? à la page 9) ;
•  La compréhension du fonctionnement du climat (processus physiques, chimiques et biologiques).
Depuis les premiers travaux du GIEC, nous disposons de beaucoup plus de données et de modèles 
climatiques plus performants. Nous

--- CHUNK 5 ---
Page : 4
• Changement Climatique 2021 : Résumé pour tous •
4 /
Le changement climatique aujourd’hui
Le réchauf

In [ ]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks_with_pages
]

In [ ]:
chunk_embeddings = model.encode(chunk_texts)

print("Nombre de chunks :", len(chunk_embeddings))
print("Dimension :", len(chunk_embeddings[0]))

Nombre de chunks : 40
Dimension : 384


In [ ]:
def retrieve(query, chunks, chunk_embeddings, model, top_k=3):

    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []

    for index in top_indices:

        results.append({
            "chunk_id": index,
            "page": chunks[index]["page"],
            "score": similarities[index],
            "text": chunks[index]["text"]
        })

    return results

In [ ]:
query = "Que sont les modèles climatiques ?"

results = retrieve(
    query,
    chunks_with_pages,
    chunk_embeddings,
    model,
    top_k=3
)

for result in results:

    print(f"\n--- CHUNK {result['chunk_id'] + 1} ---")
    print(f"Page : {result['page']}")
    print(f"Score : {result['score']:.4f}")
    print(result["text"][:500])


--- CHUNK 20 ---
Page : 9
Score : 0.6819
Que sont les modèles climatiques ?
Les modèles climatiques sont des outils informatiques que les scientifiques utilisent pour 
comprendre les changements climatiques passés, présents et futurs. Ce sont des programmes 
informatiques qui simulent le climat de la Terre, sur la base des lois fondamentales de la 
physique, de la chimie et de la biologie de l’atmosphère, de l’océan, de la glace et des terres. 
Certains modèles incluent plus de processus, de complexité et de détails que d’autres. Ainsi

--- CHUNK 7 ---
Page : 4
Score : 0.6193
Pour en savoir 
plus sur le 
changement 
climatique, vous 
pouvez lire 
la Foire aux 
questions (FAQs) 
du rapport ici : 
https://www.ipcc.ch/report/ar6/wg1/
resources/frequently-asked-questions/

--- CHUNK 4 ---
Page : 3
Score : 0.6028
t futurs (voir encadré 
Que sont les modèles climatiques ? à la page 9) ;
•  La compréhension du fonctionnement du climat (processus physiques, chimiques et biologiques).
Depuis l

In [ ]:
def rag_answer(query, top_k=3):

    results = retrieve(
        query,
        chunks_with_pages,
        chunk_embeddings,
        model,
        top_k=top_k
    )

    context = "\n\n".join(
        result["text"]
        for result in results
    )

    prompt = f"""
Tu es un assistant qui répond aux questions en utilisant uniquement
les informations fournies dans le contexte.

Si la réponse n'est pas présente dans le contexte,
dis que l'information n'est pas disponible.

Contexte :
{context}

Question :
{query}

Réponse :
"""

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    sources = [
        {
            "chunk_id": result["chunk_id"],
            "page": result["page"],
            "score": result["score"]
        }
        for result in results
    ]

    return {
        "answer": answer,
        "sources": sources
    }

In [ ]:
result = rag_answer(
    "Que sont les modèles climatiques ?"
)

print("RÉPONSE :")
print(result["answer"])

print("\nSOURCES :")

for source in result["sources"]:
    print(
        f"- Page {source['page']} "
        f"(score : {source['score']:.4f})"
    )

RÉPONSE :
Les modèles climatiques sont des outils informatiques (programmes) que les scientifiques utilisent pour comprendre les changements climatiques passés, présents et futurs. Ils simulent le climat de la Terre sur la base des lois fondamentales de la physique, de la chimie et de la biologie de l’atmosphère, de l’océan, de la glace et des terres.

Certains modèles incluent plus de processus, de complexité et de détails que d’autres. Les scientifiques les testent en comparant leurs résultats aux observations passées et aux informations paléoclimatiques ; s’ils simulent correctement les changements observés, cela donne confiance en leur capacité à représenter les processus climatiques importants. Ils permettent ensuite d’identifier les causes des changements passés et d’étudier l’évolution future du climat selon différents scénarios d’émissions de gaz à effet de serre.

SOURCES :
- Page 9 (score : 0.6819)
- Page 4 (score : 0.6193)
- Page 3 (score : 0.6028)


In [70]:
import psycopg2

connection = psycopg2.connect(
    host="localhost",
    port=5433,
    database="rag_assistant",
    user="postgres",
    password="rag_password"
)

print("Connexion réussie !")

Connexion réussie !


In [ ]:
cursor = connection.cursor()

for chunk, embedding in zip(chunks_with_pages, chunk_embeddings):

    cursor.execute(
        """
        INSERT INTO document_chunks
        (content, embedding, source, page)
        VALUES (%s, %s, %s, %s)
        """,
        (
            chunk["text"],
            embedding.tolist(),
            "document_textuel.pdf",
            chunk["page"]
        )
    )

connection.commit()

cursor.close()

print("Chunks insérés avec succès !")

Chunks insérés avec succès !


In [ ]:
query = "Que sont les modèles climatiques ?"

query_embedding = model.encode([query])[0]

print("Dimension :", len(query_embedding))

Dimension : 384


In [ ]:
cursor = connection.cursor()

cursor.execute(
    """
    SELECT
        id,
        content,
        source,
        page,
        embedding <=> %s::vector AS distance
    FROM document_chunks
    ORDER BY embedding <=> %s::vector
    LIMIT 3;
    """,
    (
        query_embedding.tolist(),
        query_embedding.tolist()
    )
)

results = cursor.fetchall()

cursor.close()

In [ ]:
for result in results:
    print("\n---")
    print("ID :", result[0])
    print("Page :", result[3])
    print("Distance :", result[4])
    print("Content :", result[1][:500])


---
ID : 20
Page : 9
Distance : 0.3180886507034302
Content : Que sont les modèles climatiques ?
Les modèles climatiques sont des outils informatiques que les scientifiques utilisent pour 
comprendre les changements climatiques passés, présents et futurs. Ce sont des programmes 
informatiques qui simulent le climat de la Terre, sur la base des lois fondamentales de la 
physique, de la chimie et de la biologie de l’atmosphère, de l’océan, de la glace et des terres. 
Certains modèles incluent plus de processus, de complexité et de détails que d’autres. Ainsi

---
ID : 7
Page : 4
Distance : 0.38068894262975683
Content : Pour en savoir 
plus sur le 
changement 
climatique, vous 
pouvez lire 
la Foire aux 
questions (FAQs) 
du rapport ici : 
https://www.ipcc.ch/report/ar6/wg1/
resources/frequently-asked-questions/

---
ID : 4
Page : 3
Distance : 0.39724475145339966
Content : t futurs (voir encadré 
Que sont les modèles climatiques ? à la page 9) ;
•  La compréhension du fonctionnement du cl

In [ ]:
def retrieve_from_db(query, model, connection, top_k=3):

    query_embedding = model.encode([query])[0]

    cursor = connection.cursor()

    cursor.execute(
        """
        SELECT
            id,
            content,
            source,
            page,
            embedding <=> %s::vector AS distance
        FROM document_chunks
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
        """,
        (
            query_embedding.tolist(),
            query_embedding.tolist(),
            top_k
        )
    )

    rows = cursor.fetchall()

    cursor.close()

    results = []

    for row in rows:

        distance = row[4]
        similarity = 1 - distance

        results.append({
            "chunk_id": row[0],
            "text": row[1],
            "source": row[2],
            "page": row[3],
            "distance": distance,
            "similarity": similarity
        })

    return results

In [ ]:
results = retrieve_from_db(
    "Que sont les modèles climatiques ?",
    model,
    connection,
    top_k=3
)

In [ ]:
for result in results:

    print("\n---")
    print("Chunk ID :", result["chunk_id"])
    print("Page :", result["page"])
    print("Distance :", result["distance"])
    print("Texte :", result["text"][:500])


---
Chunk ID : 20
Page : 9
Distance : 0.3180886507034302
Texte : Que sont les modèles climatiques ?
Les modèles climatiques sont des outils informatiques que les scientifiques utilisent pour 
comprendre les changements climatiques passés, présents et futurs. Ce sont des programmes 
informatiques qui simulent le climat de la Terre, sur la base des lois fondamentales de la 
physique, de la chimie et de la biologie de l’atmosphère, de l’océan, de la glace et des terres. 
Certains modèles incluent plus de processus, de complexité et de détails que d’autres. Ainsi

---
Chunk ID : 7
Page : 4
Distance : 0.38068894262975683
Texte : Pour en savoir 
plus sur le 
changement 
climatique, vous 
pouvez lire 
la Foire aux 
questions (FAQs) 
du rapport ici : 
https://www.ipcc.ch/report/ar6/wg1/
resources/frequently-asked-questions/

---
Chunk ID : 4
Page : 3
Distance : 0.39724475145339966
Texte : t futurs (voir encadré 
Que sont les modèles climatiques ? à la page 9) ;
•  La compréhension du fonction

In [73]:
def retrieve_from_db(
    query,
    model,
    connection,
    top_k=3,
    similarity_threshold=0.50
):

    # 1. Créer l'embedding de la question
    query_embedding = model.encode([query])[0]

    cursor = connection.cursor()

    cursor.execute(
        """
        SELECT
            id,
            content,
            source,
            page,
            embedding <=> %s::vector AS distance
        FROM document_chunks
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
        """,
        (
            query_embedding.tolist(),
            query_embedding.tolist(),
            top_k
        )
    )

    rows = cursor.fetchall()

    cursor.close()

    results = []

    for row in rows:

        distance = row[4]
        similarity = 1 - distance

        # Garder uniquement les résultats suffisamment proches
        if similarity >= similarity_threshold:

            results.append({
                "chunk_id": row[0],
                "text": row[1],
                "source": row[2],
                "page": row[3],
                "distance": distance,
                "similarity": similarity
            })

    return results

In [ ]:
result = rag_answer_from_db(
    "Que sont les modèles climatiques ?",
    model,
    connection,
    client,
    top_k=3
)

In [ ]:
print("RÉPONSE :")
print(result["answer"])

print("\nSOURCES :")

for source in result["sources"]:
    print(
        f"- {source['source']} "
        f"| Page {source['page']} "
        f"| Similarité : {source['similarity']:.4f}"
    )

RÉPONSE :
Les modèles climatiques sont des outils informatiques (des programmes informatiques) que les scientifiques utilisent pour comprendre les changements climatiques passés, présents et futurs. Ils simulent le climat de la Terre en s’appuyant sur les lois fondamentales de la physique, de la chimie et de la biologie de l’atmosphère, de l’océan, de la glace et des terres. Certains modèles intègrent davantage de processus, de complexité et de détails que d’autres, ce qui peut entraîner des variations dans les simulations ; c’est pourquoi le GIEC examine les résultats de nombreux modèles afin d’identifier les conclusions associées au plus haut degré de certitude.

SOURCES :
- document_textuel.pdf | Page 9 | Similarité : 0.6819
- document_textuel.pdf | Page 4 | Similarité : 0.6193
- document_textuel.pdf | Page 3 | Similarité : 0.6028


In [ ]:
result = rag_answer_from_db(
    "Quel est le prix d'un billet d'avion pour Paris ?",
    model,
    connection,
    client,
    top_k=3
)

print(result["answer"])



L'information demandée n'est pas disponible dans le contexte fourni.


In [ ]:
test_queries = [
    "Que sont les modèles climatiques ?",
    "Comment les activités humaines modifient-elles le climat ?",
    "Comment les océans sont-ils affectés par le changement climatique ?",
    "Pourquoi les glaciers diminuent-ils ?",
    "Quel est le prix d'un billet d'avion pour Paris ?",
    "Qui a gagné la Coupe du monde 2022 ?"
]

for query in test_queries:

    results = retrieve_from_db(
        query,
        model,
        connection,
        top_k=3
    )

    print("\n" + "=" * 70)
    print("QUESTION :", query)

    for i, result in enumerate(results, start=1):

        print(
            f"{i}. Page {result['page']} "
            f"| Similarité : {result['similarity']:.4f}"
        )


QUESTION : Que sont les modèles climatiques ?
1. Page 9 | Similarité : 0.6819
2. Page 4 | Similarité : 0.6193
3. Page 3 | Similarité : 0.6028

QUESTION : Comment les activités humaines modifient-elles le climat ?
1. Page 4 | Similarité : 0.5300
2. Page 12 | Similarité : 0.5092
3. Page 8 | Similarité : 0.5018

QUESTION : Comment les océans sont-ils affectés par le changement climatique ?
1. Page 12 | Similarité : 0.6666
2. Page 4 | Similarité : 0.6388
3. Page 16 | Similarité : 0.6338

QUESTION : Pourquoi les glaciers diminuent-ils ?
1. Page 4 | Similarité : 0.5920
2. Page 5 | Similarité : 0.5637

QUESTION : Quel est le prix d'un billet d'avion pour Paris ?

QUESTION : Qui a gagné la Coupe du monde 2022 ?


In [82]:
def rag_answer_from_db(
    query,
    model,
    connection,
    client,
    top_k=3,
    similarity_threshold=0.50
):

    # 1. Recherche dans PostgreSQL
    results = retrieve_from_db(
        query,
        model,
        connection,
        top_k=top_k,
        similarity_threshold=similarity_threshold
    )

    # 2. Aucun résultat suffisamment pertinent
    if not results:
        return {
            "answer": "L'information n'est pas disponible dans les documents.",
            "sources": []
        }

    # 3. Construire le contexte
    context = "\n\n".join(
        result["text"]
        for result in results
    )

    # 4. Construire le prompt
    prompt = f"""
Tu es un assistant qui répond aux questions en utilisant uniquement
les informations fournies dans le contexte.

Si la réponse n'est pas présente dans le contexte,
dis que l'information n'est pas disponible.

Contexte :
{context}

Question :
{query}

Réponse :
"""

    # 5. Appeler le LLM
    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    # 6. Récupérer la réponse
    answer = response.choices[0].message.content

    # 7. Préparer les sources
    sources = [
        {
            "chunk_id": result["chunk_id"],
            "source": result["source"],
            "page": result["page"],
            "similarity": result["similarity"]
        }
        for result in results
    ]

    return {
        "answer": answer,
        "sources": sources
    }

In [ ]:
result = rag_answer_from_db(
    "Que sont les modèles climatiques ?",
    model,
    connection,
    client
)

print("RÉPONSE :")
print(result["answer"])

print("\nSOURCES :")

for source in result["sources"]:
    print(
        f"- {source['source']} "
        f"| Page {source['page']} "
        f"| Similarité : {source['similarity']:.4f}"
    )

RÉPONSE :
Les modèles climatiques sont des outils informatiques que les scientifiques utilisent pour comprendre les changements climatiques passés, présents et futurs. Ce sont des programmes informatiques qui simulent le climat de la Terre, sur la base des lois fondamentales de la physique, de la chimie et de la biologie de l'atmosphère, de l'océan, de la glace et des terres.

Certains modèles incluent plus de processus, de complexité et de détails que d'autres, ce qui fait que les simulations climatiques qui en résultent peuvent varier d'un modèle à l'autre. C'est pourquoi le GIEC examine toujours les résultats de nombreux modèles climatiques afin de déterminer ceux qui sont associés au plus grand degré de certitude.

Les scientifiques testent ces modèles en comparant leurs résultats avec les observations passées et les informations paléoclimatiques (très anciennes). Si les modèles sont capables de simuler correctement les changements observés sur Terre dans le passé, cela donne confi

In [ ]:
result = rag_answer_from_db(
    "Qui a gagné la Coupe du monde 2022 ?",
    model,
    connection,
    client
)

print("RÉPONSE :")
print(result["answer"])

print("\nSOURCES :")
print(result["sources"])

RÉPONSE :
L'information n'est pas disponible dans les documents.

SOURCES :
[]


In [58]:
from pathlib import Path

documents_path = Path("../data/documents")

pdf_files = list(documents_path.glob("*.pdf"))

for pdf in pdf_files:
    print(pdf.name)

document_textuel.pdf
document_textuel_2.pdf


In [59]:
import pymupdf

all_pages = []

for pdf_path in pdf_files:

    doc = pymupdf.open(pdf_path)

    for i, page in enumerate(doc):

        text = page.get_text()

        all_pages.append({
            "source": pdf_path.name,
            "page": i + 1,
            "text": text
        })

    doc.close()

In [60]:
print("Nombre total de pages :", len(all_pages))

Nombre total de pages : 68


In [61]:
for page in all_pages[:3]:
    print("\n---")
    print("Source :", page["source"])
    print("Page :", page["page"])
    print(page["text"][:300])


---
Source : document_textuel.pdf
Page : 1
Résumé pour tous
Changement 
Climatique 2021:


---
Source : document_textuel.pdf
Page : 2
WORKING GROUP I TECHNICAL SUPPORT UNIT
Ce document n’a pas fait l’objet de la procédure de relecture officielle du GIEC.


---
Source : document_textuel.pdf
Page : 3
• Changement Climatique 2021 : Résumé pour tous •
3 /
Météo, climat et GIEC 
Quel que soit l’endroit où nous habitons, nous vivons tous avec la météo : comment les conditions 
de notre atmosphère évoluent au fil des minutes, des heures, des jours et des semaines. Nous 
vivons également tous avec le 


In [62]:
def create_chunks_with_metadata(
    pages,
    chunk_size=2000,
    chunk_overlap=300
):

    chunks = []

    for page in pages:

        source = page["source"]
        page_number = page["page"]
        text = clean_text(page["text"])

        paragraphs = text.split("\n\n")

        current_chunk = ""

        for paragraph in paragraphs:

            if len(paragraph) <= chunk_size:

                if len(current_chunk) + len(paragraph) + 2 <= chunk_size:

                    current_chunk += paragraph + "\n\n"

                else:

                    if current_chunk:

                        chunks.append({
                            "text": current_chunk.strip(),
                            "source": source,
                            "page": page_number
                        })

                    current_chunk = paragraph + "\n\n"

            else:

                if current_chunk:

                    chunks.append({
                        "text": current_chunk.strip(),
                        "source": source,
                        "page": page_number
                    })

                    current_chunk = ""

                start = 0

                while start < len(paragraph):

                    end = start + chunk_size

                    if end < len(paragraph):

                        split_position = paragraph.rfind(
                            " ",
                            start,
                            end
                        )

                        if split_position != -1:
                            end = split_position

                    chunk = paragraph[start:end]

                    chunks.append({
                        "text": chunk.strip(),
                        "source": source,
                        "page": page_number
                    })

                    start = end - chunk_overlap

        if current_chunk:

            chunks.append({
                "text": current_chunk.strip(),
                "source": source,
                "page": page_number
            })

    return chunks

In [63]:
all_chunks = create_chunks_with_metadata(
    all_pages,
    chunk_size=2000,
    chunk_overlap=300
)

In [64]:
print("Nombre total de chunks :", len(all_chunks))

Nombre total de chunks : 90


In [65]:
from collections import Counter

chunk_sources = Counter(
    chunk["source"]
    for chunk in all_chunks
)

print(chunk_sources)

Counter({'document_textuel_2.pdf': 50, 'document_textuel.pdf': 40})


In [66]:
for chunk in all_chunks[:5]:

    print("\n---")
    print("Source :", chunk["source"])
    print("Page :", chunk["page"])
    print("Texte :", chunk["text"][:300])


---
Source : document_textuel.pdf
Page : 1
Texte : Résumé pour tous
Changement 
Climatique 2021:

---
Source : document_textuel.pdf
Page : 2
Texte : WORKING GROUP I TECHNICAL SUPPORT UNIT
Ce document n’a pas fait l’objet de la procédure de relecture officielle du GIEC.

---
Source : document_textuel.pdf
Page : 3
Texte : • Changement Climatique 2021 : Résumé pour tous •
3 /
Météo, climat et GIEC 
Quel que soit l’endroit où nous habitons, nous vivons tous avec la météo : comment les conditions 
de notre atmosphère évoluent au fil des minutes, des heures, des jours et des semaines. Nous 
vivons également tous avec le 

---
Source : document_textuel.pdf
Page : 3
Texte : t futurs (voir encadré 
Que sont les modèles climatiques ? à la page 9) ;
•  La compréhension du fonctionnement du climat (processus physiques, chimiques et biologiques).
Depuis les premiers travaux du GIEC, nous disposons de beaucoup plus de données et de modèles 
climatiques plus performants. Nous

---
Source : document_

In [67]:
for chunk in all_chunks:

    if chunk["source"] == "document_textuel_2.pdf":

        print("\n---")
        print("Source :", chunk["source"])
        print("Page :", chunk["page"])
        print("Texte :", chunk["text"][:300])

        break


---
Source : document_textuel_2.pdf
Page : 1
Texte : Les énergies renouvelables
B. BELGAID 
Institut d’architecture et d’urbanisme
Université Batna 1
Juillet 2022


In [68]:
all_chunk_texts = [chunk["text"] for chunk in all_chunks]

all_chunk_embeddings = model.encode(
    all_chunk_texts,
    show_progress_bar=True
)

print("Nombre d'embeddings :", len(all_chunk_embeddings))
print("Dimension :", all_chunk_embeddings.shape[1])

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Nombre d'embeddings : 90
Dimension : 384


In [71]:
cursor = connection.cursor()

for chunk, embedding in zip(all_chunks, all_chunk_embeddings):
    cursor.execute(
        """
        INSERT INTO document_chunks
        (content, embedding, source, page)
        VALUES (%s, %s, %s, %s)
        """,
        (
            chunk["text"],
            embedding.tolist(),
            chunk["source"],
            chunk["page"]
        )
    )

connection.commit()
cursor.close()

print("Insertion terminée !")

Insertion terminée !


In [74]:
query = "Que sont les modèles climatiques ?"

results = retrieve_from_db(
    query,
    model,
    connection,
    top_k=3,
    similarity_threshold=0.50
)

for result in results:
    print("Source :", result["source"])
    print("Page :", result["page"])
    print("Similarity :", round(result["similarity"], 3))
    print("Texte :", result["text"][:300])
    print("-" * 80)

Source : document_textuel.pdf
Page : 9
Similarity : 0.682
Texte : Que sont les modèles climatiques ?
Les modèles climatiques sont des outils informatiques que les scientifiques utilisent pour 
comprendre les changements climatiques passés, présents et futurs. Ce sont des programmes 
informatiques qui simulent le climat de la Terre, sur la base des lois fondamental
--------------------------------------------------------------------------------
Source : document_textuel.pdf
Page : 4
Similarity : 0.619
Texte : Pour en savoir 
plus sur le 
changement 
climatique, vous 
pouvez lire 
la Foire aux 
questions (FAQs) 
du rapport ici : 
https://www.ipcc.ch/report/ar6/wg1/
resources/frequently-asked-questions/
--------------------------------------------------------------------------------
Source : document_textuel.pdf
Page : 3
Similarity : 0.603
Texte : t futurs (voir encadré 
Que sont les modèles climatiques ? à la page 9) ;
•  La compréhension du fonctionnement du climat (processus physiques,

In [75]:
query = "Quelles sont les différentes sources d'énergies renouvelables ?"

results = retrieve_from_db(
    query,
    model,
    connection,
    top_k=3,
    similarity_threshold=0.50
)

for result in results:
    print("Source :", result["source"])
    print("Page :", result["page"])
    print("Similarity :", round(result["similarity"], 3))
    print("Texte :", result["text"][:300])
    print("-" * 80)

Source : document_textuel_2.pdf
Page : 7
Similarity : 0.711
Texte : Les énergies 
renouvelables servent 
à produire
de la 
chaleur
de 
l’électricité
--------------------------------------------------------------------------------
Source : document_textuel_2.pdf
Page : 46
Similarity : 0.65
Texte : Chapitre VII
Programme national des énergies renouvelables
--------------------------------------------------------------------------------
Source : document_textuel_2.pdf
Page : 4
Similarity : 0.64
Texte : Les énergies renouvelables sont fournies par :
le soleil (solaire), 
le vent (éolienne), 
la chaleur de la terre (géothermique),
les barrages et cours d’eau (hydroélectricité), 
les marées (marémotrice),
la biomasse (combustibles d’origine organique).
Elles
sont
inépuisables,
contrairement
aux
--------------------------------------------------------------------------------


In [77]:
query = "Quelles sont les différentes sources d'énergies renouvelables ?"

results = retrieve_from_db(
    query,
    model,
    connection,
    top_k=3,
    similarity_threshold=0.50
)

In [78]:
context = "\n\n".join(
    result["text"]
    for result in results
)

print(context)

Les énergies 
renouvelables servent 
à produire
de la 
chaleur
de 
l’électricité

Chapitre VII
Programme national des énergies renouvelables

Les énergies renouvelables sont fournies par :
le soleil (solaire), 
le vent (éolienne), 
la chaleur de la terre (géothermique),
les barrages et cours d’eau (hydroélectricité), 
les marées (marémotrice),
la biomasse (combustibles d’origine organique).
Elles
sont
inépuisables,
contrairement
aux
sources
non
renouvelables d'origine fossile comme le pétrole, le charbon, le gaz…,
dont les gisements limités peuvent être épuisés (ils n’ont pas le temps de
se renouveler car leur formation nécessitera des millions d’années).
Les énergies fossiles sont responsables de la pollution de la planète,
ainsi que du changement climatique observé depuis quelques années.
Selon des études, les régions Est et Sud de la Méditerranée seront
parmi les régions les plus touchées par le changement climatique au
niveau mondial au cours du XXIe siècle. 1, 2
1 Plan Bleu,

In [79]:
prompt = f"""
Tu es un assistant qui répond aux questions en utilisant uniquement
les informations fournies dans le contexte.

Si la réponse n'est pas présente dans le contexte,
dis que l'information n'est pas disponible dans les documents.

Contexte :
{context}

Question :
{query}

Réponse :
"""

In [80]:
response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response.choices[0].message.content

print(answer)

Selon le contexte, les différentes sources d'énergies renouvelables sont :
- le soleil (solaire)
- le vent (éolienne)
- la chaleur de la terre (géothermique)
- les barrages et cours d’eau (hydroélectricité)
- les marées (marémotrice)
- la biomasse (combustibles d’origine organique)


In [ ]:
def rag_answer_from_db(
    query,
    model,
    connection,
    client,
    top_k=3,
    similarity_threshold=0.50
):
    # 1. Retrieval
    results = retrieve_from_db(
        query,
        model,
        connection,
        top_k=top_k,
        similarity_threshold=similarity_threshold
    )

    # 2. Aucun résultat suffisamment pertinent
    if not results:
        return {
            "answer": "L'information n'est pas disponible dans les documents.",
            "sources": []
        }

    # 3. Construction du contexte
    context = "\n\n".join(
        result["text"]
        for result in results
    )

    # 4. Construction du prompt
    prompt = f"""
Tu es un assistant qui répond aux questions en utilisant uniquement
les informations fournies dans le contexte.

Si la réponse n'est pas présente dans le contexte,
dis que l'information n'est pas disponible dans les documents.

Contexte :
{context}

Question :
{query}

Réponse :
"""

    # 5. Génération
    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    # 6. Sources
    sources = [
        {
            "chunk_id": result["chunk_id"],
            "source": result["source"],
            "page": result["page"],
            "similarity": result["similarity"]
        }
        for result in results
    ]

    return {
        "answer": answer,
        "sources": sources
    }

In [84]:
query = "Quelles sont les différentes sources d'énergies renouvelables ?"

result = rag_answer_from_db(
    query,
    model,
    connection,
    client
)

print("RÉPONSE :")
print(result["answer"])

print("\nSOURCES :")
for source in result["sources"]:
    print(
        source["source"],
        "| page", source["page"],
        "| similarity", round(source["similarity"], 3)
    )

RÉPONSE :


D'après le contexte fourni, les différentes sources d'énergies renouvelables sont :

*   Le soleil (solaire)
*   Le vent (éolienne)
*   La chaleur de la terre (géothermique)
*   Les barrages et cours d'eau (hydroélectricité)
*   Les marées (marémotrice)
*   La biomasse (combustibles d'origine organique)

SOURCES :
document_textuel_2.pdf | page 7 | similarity 0.711
document_textuel_2.pdf | page 46 | similarity 0.65
document_textuel_2.pdf | page 4 | similarity 0.64


In [85]:
query = "Quel est le prix d'un billet d'avion pour Paris ?"

result = rag_answer_from_db(
    query,
    model,
    connection,
    client
)

print("RÉPONSE :")
print(result["answer"])

print("\nSOURCES :")
for source in result["sources"]:
    print(
        source["source"],
        "| page", source["page"],
        "| similarity", round(source["similarity"], 3)
    )

RÉPONSE :
L'information n'est pas disponible dans les documents.

SOURCES :


In [86]:
context = "\n\n".join(
    f"[Source: {result['source']}, Page: {result['page']}]\n"
    f"{result['text']}"
    for result in results
)

print(context)

[Source: document_textuel_2.pdf, Page: 7]
Les énergies 
renouvelables servent 
à produire
de la 
chaleur
de 
l’électricité

[Source: document_textuel_2.pdf, Page: 46]
Chapitre VII
Programme national des énergies renouvelables

[Source: document_textuel_2.pdf, Page: 4]
Les énergies renouvelables sont fournies par :
le soleil (solaire), 
le vent (éolienne), 
la chaleur de la terre (géothermique),
les barrages et cours d’eau (hydroélectricité), 
les marées (marémotrice),
la biomasse (combustibles d’origine organique).
Elles
sont
inépuisables,
contrairement
aux
sources
non
renouvelables d'origine fossile comme le pétrole, le charbon, le gaz…,
dont les gisements limités peuvent être épuisés (ils n’ont pas le temps de
se renouveler car leur formation nécessitera des millions d’années).
Les énergies fossiles sont responsables de la pollution de la planète,
ainsi que du changement climatique observé depuis quelques années.
Selon des études, les régions Est et Sud de la Méditerranée seron

In [87]:
prompt = f"""
Tu es un assistant qui répond aux questions en utilisant uniquement
les informations fournies dans le contexte.

Règles :
- Utilise uniquement les informations présentes dans le contexte.
- Si la réponse n'est pas présente dans le contexte, dis :
  "L'information n'est pas disponible dans les documents."
- Lorsque tu réponds, cite les sources sous la forme :
  [document, page X]
- Ne crée jamais de source qui n'est pas présente dans le contexte.

Contexte :
{context}

Question :
{query}

Réponse :
"""

In [88]:
query = "Quelles sont les différentes sources d'énergies renouvelables ?"

result = rag_answer_from_db(
    query,
    model,
    connection,
    client
)

print("RÉPONSE :")
print(result["answer"])

print("\nSOURCES RETROUVÉES :")
for source in result["sources"]:
    print(
        source["source"],
        "| page", source["page"],
        "| similarity", round(source["similarity"], 3)
    )

RÉPONSE :
Les différentes sources d'énergies renouvelables sont :

- **Le soleil** (solaire)
- **Le vent** (éolienne)
- **La chaleur de la terre** (géothermique)
- **Les barrages et cours d'eau** (hydroélectricité)
- **Les marées** (marémotrice)
- **La biomasse** (combustibles d'origine organique)

Ces énergies sont inépuisables, contrairement aux sources fossiles (pétrole, charbon, gaz) dont les gisements limités peuvent être épuisés.

SOURCES RETROUVÉES :
document_textuel_2.pdf | page 7 | similarity 0.711
document_textuel_2.pdf | page 46 | similarity 0.65
document_textuel_2.pdf | page 4 | similarity 0.64
